<!-- # QRNG の一連の処理：生成 → min-entropy の評価 → 抽出 → 統計的検定 -->

# QRNG end-to-end pipeline: generation → min-entropy assessment → extraction → statistical tests

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- ほかのノートブックで 1 段階ずつ説明した処理を、1 つの流れとして通して実行します。 -->

This notebook runs, as one flow, the steps that the other notebooks explain one at a time.

<!--
1. 回路を設計する関数を定義する（2.）
2. 量子コンピュータのノイズを設定する関数を定義する（3.）
3. raw data を用意する（4.）。2.・3. の回路をシミュレーションするか、保存した raw data のファイルを読み込むかを選ぶ
4. SP 800-90B の公式ツール `ea_non_iid` で min-entropy を推定する（5.）
5. 推定した min-entropy で Toeplitz または Trevisan の抽出器を適用する（6.）
6. 抽出したビット列を SP 800-22 の公式ツール STS で検定する（7.）
7. 結果をまとめる（8.）
-->

1. Define the function that designs the circuit (2.)
2. Define the function that sets the noise of the quantum computer (3.)
3. Prepare the raw data (4.): either simulate the circuit of 2. and 3., or load a saved raw-data file
4. Estimate the min-entropy with `ea_non_iid`, the official SP 800-90B tool (5.)
5. Apply the Toeplitz or Trevisan extractor with the estimated min-entropy (6.)
6. Test the extracted bit string with STS, the official SP 800-22 tool (7.)
7. Summarize the results (8.)

```
 circuit (2.) + noise (3.)         raw-data file
            │ simulation                │
            └──────────┬────────────────┘
                       ▼
                 raw bits (4.)
                       │ first H90B_SAMPLES bits
                       ▼
          SP 800-90B: min-entropy H_I (5.)
                       │ h = H_I
                       ▼
       Toeplitz / Trevisan extractor (6.)
                       │ N_SEQUENCES × SEQUENCE_LENGTH bits
                       ▼
             SP 800-22: STS (7.)
```

<!-- ### 準備するもの -->

### Prerequisites

<!--
- `README.md` の手順で Python の環境を作り、公式ツール（`ea_non_iid` と STS）をこのノートブックと同じフォルダにビルドしてください。ツールがない場合、その段階は実行せず、そのことを表示します。
- このノートブックは、同じフォルダにある教材のノートブック（`SP800_90B_textbook.ipynb`、`Toeplitz_extractor_textbook.ipynb`、`Trevisan_extractor_textbook.ipynb`、`SP800_22_textbook.ipynb`）から、関数の定義だけを読み込んで使います（1.）。それぞれの関数のしくみと注意点は、教材のノートブックで説明しています。
-->

- Set up the Python environment and build the official tools (`ea_non_iid` and STS) in the same folder as this notebook by following `README.md`. If a tool is missing, that step is skipped and a message says so.
- This notebook loads only the function definitions from the textbook notebooks in the same folder (`SP800_90B_textbook.ipynb`, `Toeplitz_extractor_textbook.ipynb`, `Trevisan_extractor_textbook.ipynb`, `SP800_22_textbook.ipynb`) and uses them (1.). How each function works, and its caveats, are explained in the textbook notebooks.

<!-- ## 使い方 -->

## How to use this notebook

<!-- 上のセルから順に実行してください。条件は、各節の最初のセルの変数を書き換えて選びます。 -->

Run the cells in order from the top. The conditions are selected by editing the variables at the top of each section.

<!--
| 変数 | 節 | 意味 | 既定値 |
|---|---|---|---|
| `SAVE_RESULTS` | 1. | 結果を保存するか（`True` なら `results/pipeline/<日時>/` に保存） | `False` |
| `CIRCUIT`, `GATE` | 2. | 回路（`"C1"`, `"C2"`, `"C3"`）と、重ね合わせを作るゲート（`"H"`, `"Rx"`, `"Ry"`） | `"C1"`, `"H"` |
| `NOISE` | 3. | ノイズの成分と強さの辞書（`{}` ならノイズなし） | 脱分極・CZ の脱分極・熱緩和・読み出し誤差（C1〜C3 のノートブックの `"both"` と同じ値） |
| `SOURCE` | 4. | `"simulation"`（2.・3. の回路をシミュレーション）または `"file"`（`RAW_FILE` を読み込む） | `"simulation"` |
| `SHOTS`, `SEED` | 4. | シミュレーションの shot 数と乱数シード | `70_000_000`, `None` |
| `RAW_FILE` | 4. | `SOURCE = "file"` のときに読み込む、`'0'` と `'1'` の文字のファイル | `None` |
| `H90B_SAMPLES`, `H_SUBMITTER` | 5. | `ea_non_iid` に渡すサンプル数（raw bits の先頭から）と、提出者の推定値 | `1_000_000`, `None` |
| `EXTRACTOR`, `EPSILON`, `SEED_RNG`, `H_MIN_OVERRIDE` | 6. | 抽出器（`"toeplitz"` か `"trevisan"`）、1 ブロックあたりの誤差、シードの作り方、5. の値の代わりに使う min-entropy | `"toeplitz"`, `2.0 ** -100`, `None`, `None` |
| `TOEPLITZ_BLOCK_SIZE`, `TREVISAN_BLOCK_SIZE` | 6. | 1 ブロックのビット数 | `1_000_000`, `10_000` |
| `N_SEQUENCES`, `SEQUENCE_LENGTH` | 7. | STS で検定する系列の本数と長さ | `55`, `1_000_000` |
-->

| Variable | Section | Meaning | Default |
|---|---|---|---|
| `SAVE_RESULTS` | 1. | Whether to save the results (`True` saves them in `results/pipeline/<date_time>/`) | `False` |
| `CIRCUIT`, `GATE` | 2. | The circuit (`"C1"`, `"C2"`, `"C3"`) and the gate that creates the superposition (`"H"`, `"Rx"`, `"Ry"`) | `"C1"`, `"H"` |
| `NOISE` | 3. | Dictionary of noise components and strengths (`{}` means no noise) | Depolarizing, CZ depolarizing, thermal relaxation, and readout error (the same values as `"both"` in the C1–C3 notebooks) |
| `SOURCE` | 4. | `"simulation"` (simulate the circuit of 2. and 3.) or `"file"` (load `RAW_FILE`) | `"simulation"` |
| `SHOTS`, `SEED` | 4. | Number of shots and random seed of the simulation | `70_000_000`, `None` |
| `RAW_FILE` | 4. | File of the characters `'0'` and `'1'` loaded when `SOURCE = "file"` | `None` |
| `H90B_SAMPLES`, `H_SUBMITTER` | 5. | Number of samples passed to `ea_non_iid` (from the start of the raw bits) and the submitter's estimate | `1_000_000`, `None` |
| `EXTRACTOR`, `EPSILON`, `SEED_RNG`, `H_MIN_OVERRIDE` | 6. | Extractor (`"toeplitz"` or `"trevisan"`), error per block, how the seed is generated, and a min-entropy to use instead of the value from 5. | `"toeplitz"`, `2.0 ** -100`, `None`, `None` |
| `TOEPLITZ_BLOCK_SIZE`, `TREVISAN_BLOCK_SIZE` | 6. | Number of bits per block | `1_000_000`, `10_000` |
| `N_SEQUENCES`, `SEQUENCE_LENGTH` | 7. | Number and length of the sequences tested by STS | `55`, `1_000_000` |

<!--
**必要なビット数と実行時間の目安**：SP 800-22 の標準の大きさ（$10^6$ ビット × 55 本）で検定するには、抽出後に $5.5 \times 10^7$ ビットが必要です。抽出で 1 ビットあたり min-entropy 程度に減るので、raw bits は約 $5.5 \times 10^7 / h$ ビット要ります（C1 回路・既定のノイズで $h \approx 0.86$ なら約 $6.4 \times 10^7$ ビット。既定の `SHOTS` は余裕を見て $7 \times 10^7$ にしています）。既定の設定では、シミュレーションに 1〜2 分、`ea_non_iid` に数十秒、Toeplitz の抽出に 1 分ほど、STS に 3〜5 分かかります。

- **C2 回路**は 1 shot で 5 ビット、**C3 回路**は 1 shot で 1 ビット（多数決）なので、`SHOTS` を回路に合わせて決めてください。必要な raw bits と shot 数は 5. で表示します。
- **Trevisan の抽出器**は Toeplitz より遅く、$5.5 \times 10^7$ ビットの抽出に 40 分ほどかかります（1 万ビットのブロック 1 つに約 0.35 秒）。試すときは `N_SEQUENCES` を小さくしてください（6. で、必要な分のブロックだけを抽出します）。
- 動作を確かめるだけなら、`SHOTS` と `N_SEQUENCES` を小さくしてください。系列が 55 本より少ないと、SP 800-22 の判定は目安になります。
-->

**Required number of bits and approximate run time**: testing at the standard size of SP 800-22 ($10^6$ bits × 55 sequences) needs $5.5 \times 10^7$ bits after extraction. Extraction reduces the bits to roughly the min-entropy per bit, so about $5.5 \times 10^7 / h$ raw bits are needed (about $6.4 \times 10^7$ bits for the C1 circuit with the default noise, $h \approx 0.86$; the default `SHOTS` is $7 \times 10^7$ to leave a margin). With the default settings, the simulation takes 1–2 minutes, `ea_non_iid` a few tens of seconds, the Toeplitz extraction about 1 minute, and STS 3–5 minutes.

- The **C2 circuit** gives 5 bits per shot and the **C3 circuit** 1 bit per shot (majority vote), so choose `SHOTS` for your circuit. The required numbers of raw bits and shots are shown in 5.
- **Trevisan's extractor** is slower than Toeplitz; extracting $5.5 \times 10^7$ bits takes about 40 minutes (about 0.35 s per block of 10,000 bits). To try it, reduce `N_SEQUENCES` (6. extracts only as many blocks as needed).
- To just check that everything works, reduce `SHOTS` and `N_SEQUENCES`. With fewer than 55 sequences, the SP 800-22 judgment is only a guide.

<!-- ### 保存されるファイル -->

### Saved files

<!-- 1. のセルで `SAVE_RESULTS = True` にすると、そのセルを実行した日時の名前のフォルダ `results/pipeline/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作り、次のファイルを保存します。`False`（既定）では何も保存しません。1. のセルを実行し直すたびに新しいフォルダになります。 -->

If you set `SAVE_RESULTS = True` in the cell in 1., a folder named after the date and time that cell was run, `results/pipeline/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created and the following files are saved in it. With `False` (the default), nothing is saved. Each time you rerun the cell in 1., a new folder is used.

<!--
```
results/pipeline/
└── 2026-11-10_153012/              RUN_DIR（1. のセルを実行した日時）
    ├── raw_bits.txt                 raw bits（シミュレーションのときだけ）
    ├── raw_bits.json                raw data の条件（回路・ノイズ・shot 数・シード、またはファイルのパスと SHA-256）
    ├── sp800_90b_input.bin          ea_non_iid に渡したファイル（1 サンプル 1 バイト）
    ├── sp800_90b_log.txt            ea_non_iid の出力の全文
    ├── extracted_bits.txt           抽出したビット列
    ├── extraction_seed.txt          抽出器のシード
    ├── extraction_info.json         抽出のパラメータ
    ├── sts_results/                 STS の結果のフォルダのコピー
    ├── sts_report.csv               STS の報告の 188 行
    └── summary.json                 全体のまとめ（8. の表と同じ内容）
```
-->

```
results/pipeline/
└── 2026-11-10_153012/              RUN_DIR (date and time the cell in 1. was run)
    ├── raw_bits.txt                 raw bits (only for the simulation)
    ├── raw_bits.json                conditions of the raw data (circuit, noise, shots, seed, or the file path and SHA-256)
    ├── sp800_90b_input.bin          file passed to ea_non_iid (one sample per byte)
    ├── sp800_90b_log.txt            full output of ea_non_iid
    ├── extracted_bits.txt           extracted bit string
    ├── extraction_seed.txt          seed of the extractor
    ├── extraction_info.json         parameters of the extraction
    ├── sts_results/                 copy of the STS results folder
    ├── sts_report.csv               the 188 rows of the STS report
    └── summary.json                 summary of the whole run (same as the table in 8.)
```

<!-- `.txt` のビット列は、`'0'` と `'1'` の文字を改行なしで並べたもので、教材のノートブックの `load_bits` で読めます。CSV の列は SP800_22 のノートブックの `<data_label>_sts.csv` と同じです。ファイルは上書きしません。 -->

The `.txt` bit strings are the characters `'0'` and `'1'` with no line breaks, and can be read with `load_bits` in the textbook notebooks. The CSV columns are the same as `<data_label>_sts.csv` in the SP800_22 notebook. Files are never overwritten.

<!-- ### 但し書き -->

### Caveats

<!--
- **min-entropy は raw bits の先頭だけで推定します**：5. は先頭の `H90B_SAMPLES` ビットで推定し、その値を raw bits 全体に使います。raw data の性質が途中で変わらないことを前提にしています。
- **シミュレーションは真の乱数ではありません**：シミュレータの結果は古典コンピュータの擬似乱数から作られます。このノートブックは処理の流れを確かめるためのもので、得られたビット列を暗号に使ってはいけません。
- **抽出器のシード**：`SEED_RNG = None` なら OS の暗号用乱数で作ります。整数にすると誰でも再現できる疑似乱数になるので、確認用にだけ使ってください。ブロックごとの抽出では同じシードをすべてのブロックに使い、全体の誤差はブロック数 × `EPSILON` です（Toeplitz のノートブックの手法 D）。
-->

- **The min-entropy is estimated from the start of the raw bits only**: 5. estimates it from the first `H90B_SAMPLES` bits and uses that value for all raw bits. This assumes that the properties of the raw data do not change along the way.
- **Simulations are not true random numbers**: simulator results come from a classical computer's pseudorandom numbers. This notebook is for checking the flow of the processing; the resulting bit strings must not be used for cryptography.
- **Seed of the extractor**: with `SEED_RNG = None`, it is generated from the OS's cryptographic random numbers. An integer gives pseudorandom numbers that anyone can reproduce, so use it only for checking. Block-by-block extraction uses the same seed for all blocks, and the total error is the number of blocks × `EPSILON` (Method D of the Toeplitz notebook).

<!-- ## 1. 準備 -->

## 1. Setup

<!-- ライブラリを読み込み、結果を保存するかどうか `SAVE_RESULTS` を決めます。そのあと、教材のノートブックから関数の定義だけを読み込みます。 -->

We load the libraries and choose whether to save the results, `SAVE_RESULTS`. Then we load only the function definitions from the textbook notebooks.

<!-- `load_definitions` は、ノートブックのコードセルのうち、`import`・関数の定義・大文字の名前の定数（と、それだけを含む `if`・`try`）だけを実行し、ほかのセル（実験を実行する部分）は実行しません。読み込んだ関数は、`sp90b.run_ea_non_iid(...)` のように、ノートブックごとの名前を付けて使います。関数のしくみは各ノートブックを参照してください。 -->

`load_definitions` runs only the `import` statements, function definitions, and constants with UPPER_CASE names (and `if`/`try` blocks containing only those) from a notebook's code cells, and does not run the rest (the parts that run experiments). The loaded functions are used with a prefix per notebook, as in `sp90b.run_ea_non_iid(...)`. For how the functions work, see each notebook.

<!--
| 名前 | 読み込むノートブック | 主に使う関数 |
|---|---|---|
| `sp90b` | `SP800_90B_textbook.ipynb` | `write_90b_input`, `run_ea_non_iid`, `show_estimates`, `tool_version` |
| `toeplitz` | `Toeplitz_extractor_textbook.ipynb` | `load_bits`, `save_bits`, `mcv_min_entropy`, `calculate_output_length`, `toeplitz_extraction_blocks` |
| `trevisan` | `Trevisan_extractor_textbook.ipynb` | `trevisan_extraction_numba`, `trevisan_extraction` |
| `sp22` | `SP800_22_textbook.ipynb` | `split_sequences`, `write_sts_input`, `run_sts`, `parse_sts_report`, `show_sts_summary`, `show_flagged_rows`, `save_sts_csv` |
-->

| Name | Notebook loaded | Main functions used |
|---|---|---|
| `sp90b` | `SP800_90B_textbook.ipynb` | `write_90b_input`, `run_ea_non_iid`, `show_estimates`, `tool_version` |
| `toeplitz` | `Toeplitz_extractor_textbook.ipynb` | `load_bits`, `save_bits`, `mcv_min_entropy`, `calculate_output_length`, `toeplitz_extraction_blocks` |
| `trevisan` | `Trevisan_extractor_textbook.ipynb` | `trevisan_extraction_numba`, `trevisan_extraction` |
| `sp22` | `SP800_22_textbook.ipynb` | `split_sequences`, `write_sts_input`, `run_sts`, `parse_sts_report`, `show_sts_summary`, `show_flagged_rows`, `save_sts_csv` |

In [ ]:
import ast
import hashlib
import json
import math
import tempfile
import time
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import qiskit
import qiskit_aer
from IPython.display import Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error

print("qiskit     :", qiskit.__version__)
print("qiskit-aer :", qiskit_aer.__version__)

# ---- Saving (see "Saved files") ----
SAVE_RESULTS = False
RUN_DIR = Path("results/pipeline") / datetime.now().strftime("%Y-%m-%d_%H%M%S") if SAVE_RESULTS else None
# Folder for working files (the inputs of the tools): RUN_DIR when saving, otherwise a temporary folder
WORK_DIR = RUN_DIR if RUN_DIR is not None else Path(tempfile.mkdtemp(prefix="qrng_pipeline_"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("save to    :", RUN_DIR if RUN_DIR is not None else "nothing is saved")


def is_definition(node):
    # import, def, UPPER_CASE constants, and if/try blocks that contain only those
    if isinstance(node, (ast.Import, ast.ImportFrom, ast.FunctionDef)):
        return True
    if isinstance(node, ast.Assign):
        names = [n.id for t in node.targets for n in ast.walk(t) if isinstance(n, ast.Name)]
        return all(name.isupper() for name in names)
    if isinstance(node, (ast.If, ast.Try)):
        inner = node.body + node.orelse + getattr(node, "finalbody", [])
        inner += [s for handler in getattr(node, "handlers", []) for s in handler.body]
        return all(is_definition(s) for s in inner)
    return False


def load_definitions(notebook):
    # Run only the definitions of a textbook notebook in a separate namespace and return it
    namespace = {"__name__": Path(notebook).stem}
    for cell in json.loads(Path(notebook).read_text(encoding="utf-8"))["cells"]:
        if cell["cell_type"] == "code":
            tree = ast.parse("".join(cell["source"]))
            tree.body = [node for node in tree.body if is_definition(node)]
            exec(compile(tree, notebook, "exec"), namespace)
    return SimpleNamespace(**namespace)


sp90b = load_definitions("SP800_90B_textbook.ipynb")
toeplitz = load_definitions("Toeplitz_extractor_textbook.ipynb")
trevisan = load_definitions("Trevisan_extractor_textbook.ipynb")
sp22 = load_definitions("SP800_22_textbook.ipynb")

print("ea_non_iid :", "available" if sp90b.TOOL_AVAILABLE else f"not found ({sp90b.EA_NON_IID})")
print("STS        :", "available" if sp22.STS_AVAILABLE else f"not found ({sp22.STS_DIR})")
print("numba      :", "available" if trevisan.NUMBA_AVAILABLE else "not available (Trevisan uses the slow Python version)")

<!-- ## 2. 回路を設計する関数 -->

## 2. Function that designs the circuit

<!-- `build_circuit()` が測定までを含む回路を、`shot_to_bits(s)` が 1 shot の測定結果の文字列 `s` から出力のビットを作ります。`CIRCUIT` で C1〜C3 の回路を選べます。自分の回路を使うときは、この 2 つの関数を書き換えてください（ゲートは `BASIS_GATES` に変換されます）。 -->

`build_circuit()` returns the circuit including measurement, and `shot_to_bits(s)` makes the output bits from the measured string `s` of one shot. `CIRCUIT` selects the C1–C3 circuits. To use your own circuit, rewrite these two functions (the gates are converted to `BASIS_GATES`).

<!--
| `CIRCUIT` | 回路 | 1 shot の出力 |
|---|---|---|
| `"C1"` | 1 量子ビットに `GATE` をかけて測定する | 1 ビット |
| `"C2"` | 5 量子ビットのそれぞれに `GATE` をかけて測定する | 5 ビット（Qiskit の順：量子ビット 4, 3, 2, 1, 0 を連結） |
| `"C3"` | 量子ビット 2 に `GATE` をかけ、CX ゲートで 5 量子ビットの GHZ 状態にして測定する | 1 ビット（5 ビットの多数決） |
-->

| `CIRCUIT` | Circuit | Output per shot |
|---|---|---|
| `"C1"` | Applies `GATE` to 1 qubit and measures | 1 bit |
| `"C2"` | Applies `GATE` to each of 5 qubits and measures | 5 bits (Qiskit order: qubits 4, 3, 2, 1, 0, joined) |
| `"C3"` | Applies `GATE` to qubit 2, makes a 5-qubit GHZ state with CX gates, and measures | 1 bit (majority vote of the 5 bits) |

In [ ]:
CIRCUIT = "C1"    # "C1", "C2", "C3"
GATE = "H"        # "H", "Rx", "Ry"

BASIS_GATES = ["rx", "ry", "cz"]                 # Native gates of IQM Spark (Odra 5)
N_QUBITS = 1 if CIRCUIT == "C1" else 5
CONTROL, TARGETS = 2, [1, 3, 0, 4]               # C3: central qubit and CX targets (Fig. 4 of the paper)


def add_superposition(qc, qubit):
    # One gate that creates an equal superposition from |0>
    if GATE == "H":
        qc.h(qubit)
    elif GATE == "Rx":
        qc.rx(math.pi / 2, qubit)
    elif GATE == "Ry":
        qc.ry(math.pi / 2, qubit)
    else:
        raise ValueError(f"GATE must be 'H', 'Rx', or 'Ry' (got: {GATE!r})")


def build_circuit():
    qc = QuantumCircuit(N_QUBITS, N_QUBITS)
    if CIRCUIT == "C1":
        add_superposition(qc, 0)
    elif CIRCUIT == "C2":
        for q in range(N_QUBITS):
            add_superposition(qc, q)
    elif CIRCUIT == "C3":
        add_superposition(qc, CONTROL)
        for target in TARGETS:
            qc.cx(CONTROL, target)
    else:
        raise ValueError(f"CIRCUIT must be 'C1', 'C2', or 'C3' (got: {CIRCUIT!r})")
    qc.measure(range(N_QUBITS), range(N_QUBITS))
    return qc


def shot_to_bits(s):
    # Output bits of one shot from its measured string s (Qiskit order: q_{n-1} ... q_0)
    if CIRCUIT == "C3":
        return "1" if s.count("1") > N_QUBITS // 2 else "0"   # majority vote
    return s                                                    # C1: 1 bit, C2: the 5 bits joined


BITS_PER_SHOT = len(shot_to_bits("0" * N_QUBITS))
qc_t = transpile(build_circuit(), basis_gates=BASIS_GATES, optimization_level=0)
print(qc_t.draw(fold=-1))
print("bits per shot:", BITS_PER_SHOT)

<!-- ## 3. 量子コンピュータのノイズを設定する関数 -->

## 3. Function that sets the noise of the quantum computer

<!-- C1〜C3 のノートブックの 5.2 と同じしくみです。`NOISE_COMPONENTS` に成分ごとの誤差の作り方を、`NOISE` に使う成分と強さを書き、`build_noise_model(NOISE)` で `NoiseModel` を作ります。成分の追加のしかたと、Qiskit Aer で加えられるノイズの表は、C1〜C3 のノートブックの 5.2 を参照してください。 -->

This is the same mechanism as 5.2 in the C1–C3 notebooks. `NOISE_COMPONENTS` describes how each component creates its errors, `NOISE` lists the components to use and their strengths, and `build_noise_model(NOISE)` builds the `NoiseModel`. For how to add components and the table of noise that Qiskit Aer can add, see 5.2 in the C1–C3 notebooks.

<!--
| 成分 | 加える誤差 | 強さ |
|---|---|---|
| `depolarizing` | `rx`, `ry` の後に `depolarizing_error(λ, 1)` | λ（0〜1） |
| `cz_depolarizing` | `cz` の後に `depolarizing_error(λ_CZ, 2)`（C3 回路だけで効く） | $\lambda_{\mathrm{CZ}}$（0〜1） |
| `thermal` | `rx`, `ry` の後に `thermal_relaxation_error(T1, T2, 倍率 × GATE_TIME)`、`cz` の後にその 2 量子ビット版 | ゲート時間の倍率（1 が実機の値） |
| `readout` | 対称な読み出し誤差 | ε（0〜1） |
| `readout_asym` | $P(1\mid 0) = 0$, $P(0\mid 1) = \delta$ の読み出し誤差 | δ（0〜1） |
-->

| Component | Error added | Strength |
|---|---|---|
| `depolarizing` | `depolarizing_error(λ, 1)` after `rx`, `ry` | λ (0–1) |
| `cz_depolarizing` | `depolarizing_error(λ_CZ, 2)` after `cz` (only matters for the C3 circuit) | $\lambda_{\mathrm{CZ}}$ (0–1) |
| `thermal` | `thermal_relaxation_error(T1, T2, multiplier × GATE_TIME)` after `rx`, `ry`, and its 2-qubit version after `cz` | Gate-time multiplier (1 is the hardware value) |
| `readout` | Symmetric readout error | ε (0–1) |
| `readout_asym` | Readout error with $P(1\mid 0) = 0$, $P(0\mid 1) = \delta$ | δ (0–1) |

In [ ]:
T1, T2 = 0.964e6, 1.155e6          # Relaxation and dephasing times (ns)
GATE_TIME, CZ_GATE_TIME = 20, 60   # Gate times (ns)

# Components used and their strengths ({} means no noise). These are the values of "both" in the C1-C3 notebooks.
NOISE = {"depolarizing": 0.00198, "cz_depolarizing": 0.013245, "thermal": 1, "readout": 0.03}

NOISE_COMPONENTS = {
    "depolarizing": dict(max=1, gate_errors=lambda s: {("rx", "ry"): depolarizing_error(s, 1)}),
    "cz_depolarizing": dict(max=1, gate_errors=lambda s: {("cz",): depolarizing_error(s, 2)}),
    "thermal": dict(max=None, gate_errors=lambda s: {
        ("rx", "ry"): thermal_relaxation_error(T1, T2, s * GATE_TIME),
        ("cz",): thermal_relaxation_error(T1, T2, s * CZ_GATE_TIME).tensor(
            thermal_relaxation_error(T1, T2, s * CZ_GATE_TIME))}),
    "readout": dict(max=1, readout=lambda s: [[1 - s, s], [s, 1 - s]]),
    "readout_asym": dict(max=1, readout=lambda s: [[1, 0], [s, 1 - s]]),
}


def build_noise_model(noise):
    # noise: {component: strength}. Errors on the same gate are composed in the order written;
    # readout errors are combined into one confusion matrix (a second ReadoutError would replace the first).
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    gate_errors, confusion = {}, np.eye(2)
    for name, strength in noise.items():
        spec = NOISE_COMPONENTS[name]
        if strength < 0 or (spec["max"] is not None and strength > spec["max"]):
            raise ValueError(f"The strength of {name!r} is out of range: {strength}")
        if strength == 0:
            continue
        for gates, error in spec.get("gate_errors", lambda s: {})(strength).items():
            for gate in gates:
                gate_errors[gate] = error if gate not in gate_errors else gate_errors[gate].compose(error)
        if "readout" in spec:
            confusion = confusion @ np.array(spec["readout"](strength), dtype=float)
    for gate, error in gate_errors.items():
        noise_model.add_all_qubit_quantum_error(error, [gate])
    if not np.allclose(confusion, np.eye(2)):
        noise_model.add_all_qubit_readout_error(ReadoutError(confusion))
    return noise_model


noise_model = build_noise_model(NOISE)
print(noise_model)

<!-- ## 4. raw data を用意する -->

## 4. Preparing the raw data

<!-- `SOURCE = "simulation"` なら 2.・3. の回路を `SHOTS` 回シミュレーションし、`SOURCE = "file"` なら `RAW_FILE`（`'0'` と `'1'` の文字のファイル。C1〜C3 のノートブックで保存した raw data など）を読み込みます。シミュレーションは、メモリを使いすぎないように $10^6$ shot ずつに分けて行い、塊 $k$ にはシード `SEED + k` を使います。 -->

With `SOURCE = "simulation"`, the circuit of 2. and 3. is simulated `SHOTS` times; with `SOURCE = "file"`, `RAW_FILE` (a file of the characters `'0'` and `'1'`, such as raw data saved by the C1–C3 notebooks) is loaded. To avoid using too much memory, the simulation runs in chunks of $10^6$ shots, and chunk $k$ uses the seed `SEED + k`.

In [ ]:
SOURCE = "simulation"   # "simulation" or "file"
SHOTS = 70_000_000      # Number of shots of the simulation
SEED = None             # Random seed of the simulation (None: different results each time)
RAW_FILE = None         # e.g., Path("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")
CHUNK = 1_000_000       # Shots per chunk of the simulation


def simulate_bits(shots, seed=None):
    # Simulate the circuit of 2. with the noise of 3. and return the output bits in measurement order
    backend = AerSimulator(noise_model=noise_model)
    parts = []
    for k, start in enumerate(range(0, shots, CHUNK)):
        n = min(CHUNK, shots - start)
        chunk_seed = None if seed is None else seed + k
        memory = backend.run(qc_t, shots=n, memory=True, seed_simulator=chunk_seed).result().get_memory()
        text = "".join(shot_to_bits(s) for s in memory)
        parts.append(np.frombuffer(text.encode("ascii"), dtype=np.uint8) - ord("0"))
    return np.concatenate(parts)


started = time.time()
if SOURCE == "simulation":
    raw_bits = simulate_bits(SHOTS, SEED)
    raw_info = dict(source="simulation", circuit=CIRCUIT, gate=GATE, noise=NOISE, shots=SHOTS, seed=SEED,
                    bits_per_shot=BITS_PER_SHOT, qiskit_version=qiskit.__version__,
                    qiskit_aer_version=qiskit_aer.__version__)
elif SOURCE == "file":
    raw_bits = toeplitz.load_bits(RAW_FILE)
    raw_info = dict(source="file", path=str(RAW_FILE),
                    sha256=hashlib.sha256(Path(RAW_FILE).read_bytes()).hexdigest())
else:
    raise ValueError(f"SOURCE must be 'simulation' or 'file' (got: {SOURCE!r})")
raw_info.update(n_bits=int(raw_bits.size), P1=float(raw_bits.mean()))
print(f"raw bits: {raw_bits.size:,} bits, P(1) = {raw_bits.mean():.5f} ({time.time() - started:.0f} s)")

if RUN_DIR is not None:
    if SOURCE == "simulation":
        toeplitz.save_bits(RUN_DIR / "raw_bits.txt", raw_bits)
    (RUN_DIR / "raw_bits.json").write_text(json.dumps(raw_info, indent=2) + "\n", encoding="utf-8")
    print("Saved:", RUN_DIR / "raw_bits.json")

<!-- ## 5. SP 800-90B で min-entropy を推定する -->

## 5. Estimating the min-entropy with SP 800-90B

<!-- raw bits の先頭 `H90B_SAMPLES` ビットを 1 サンプル 1 バイトのファイルにして `ea_non_iid -i -v` を実行し、10 種類の推定値の最小値 $H_{\text{original}}$ から初期エントロピー推定 $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ を求めます（SP800_90B のノートブックの 4.・5.）。`ea_non_iid` がない場合は、代わりに MCV 推定を使い、そのことを表示します。 -->

We write the first `H90B_SAMPLES` raw bits to a file with one sample per byte, run `ea_non_iid -i -v`, and obtain the initial entropy estimate $H_I = \min(H_{\text{original}}, H_{\text{submitter}})$ from the minimum of the 10 estimates, $H_{\text{original}}$ (4. and 5. of the SP800_90B notebook). If `ea_non_iid` is not available, the MCV estimate is used instead and a message says so.

<!-- 最後に、SP 800-22 の標準の大きさ（$10^6$ ビット × 55 本）の検定に必要な raw bits の目安（$5.5 \times 10^7 / H_I$）と shot 数を表示します。 -->

Finally, the cell shows the approximate raw bits ($5.5 \times 10^7 / H_I$) and shots needed to test at the standard size of SP 800-22 ($10^6$ bits × 55 sequences).

In [ ]:
H90B_SAMPLES = 1_000_000   # Samples passed to ea_non_iid (SP 800-90B requires at least 10^6)
H_SUBMITTER = None         # Submitter's estimate (None: not used)

samples = raw_bits[:H90B_SAMPLES]
if samples.size < 1_000_000:
    print("Note: SP 800-90B requires at least 10^6 samples.")
input_path = WORK_DIR / "sp800_90b_input.bin"
input_sha = sp90b.write_90b_input(input_path, samples)

if sp90b.TOOL_AVAILABLE:
    result_90b = sp90b.run_ea_non_iid(input_path)
    sp90b.show_estimates({"raw bits": result_90b})
    h_source = "SP 800-90B (ea_non_iid)"
    H_original = result_90b["H_original"]
    if RUN_DIR is not None:
        (RUN_DIR / "sp800_90b_log.txt").write_text(result_90b["stdout"], encoding="utf-8")
else:
    print("ea_non_iid is not available, so the MCV estimate is used instead.")
    h_source = "MCV estimate (ea_non_iid not available)"
    H_original = toeplitz.mcv_min_entropy(samples)
H_I = H_original if H_SUBMITTER is None else min(H_original, H_SUBMITTER)
print(f"H_original = {H_original:.6f}, H_I = {H_I:.6f} ({h_source})")

needed_raw = 55 * 1_000_000 / H_I
print(f"Raw bits needed for 55 sequences of 10^6 bits after extraction ≈ {needed_raw:,.0f}"
      f" (= {math.ceil(needed_raw / BITS_PER_SHOT):,} shots); available: {raw_bits.size:,}")

<!-- ## 6. 抽出器を適用する -->

## 6. Applying the extractor

<!-- 5. の $H_I$（`H_MIN_OVERRIDE` を指定すればその値）を 1 ビットあたりの min-entropy $h$ として、raw bits をブロックに分け、すべてのブロックに同じシードを使って抽出します。7. の検定に必要な `N_SEQUENCES × SEQUENCE_LENGTH` ビットが得られる分のブロックだけを抽出し、残りの raw bits は使いません。 -->

Using $H_I$ from 5. (or `H_MIN_OVERRIDE` if given) as the min-entropy per bit $h$, we split the raw bits into blocks and extract from them using the same seed for all blocks. Only as many blocks as needed to obtain the `N_SEQUENCES × SEQUENCE_LENGTH` bits for the tests in 7. are extracted; the remaining raw bits are not used.

<!--
| `EXTRACTOR` | 使う関数 | 1 ブロックの出力 |
|---|---|---|
| `"toeplitz"` | `toeplitz.toeplitz_extraction_blocks`（Toeplitz のノートブックの 6.1） | $m_b = \lfloor n_b h + 2\log_2 \varepsilon \rfloor$ |
| `"trevisan"` | `trevisan.trevisan_extraction_numba`（`numba` がなければ `trevisan_extraction`。Trevisan のノートブックの 7.） | $k \ge m + 4\log_2(m/\varepsilon) + 6$ を満たす最大の $m$ |
-->

| `EXTRACTOR` | Function used | Output per block |
|---|---|---|
| `"toeplitz"` | `toeplitz.toeplitz_extraction_blocks` (6.1 of the Toeplitz notebook) | $m_b = \lfloor n_b h + 2\log_2 \varepsilon \rfloor$ |
| `"trevisan"` | `trevisan.trevisan_extraction_numba` (`trevisan_extraction` if `numba` is not available; 7. of the Trevisan notebook) | The largest $m$ satisfying $k \ge m + 4\log_2(m/\varepsilon) + 6$ |

In [ ]:
EXTRACTOR = "toeplitz"          # "toeplitz" or "trevisan"
EPSILON = 2.0 ** -100           # Error per block
SEED_RNG = None                 # None: the OS's cryptographic random numbers; integer: reproducible (for checking only)
H_MIN_OVERRIDE = None           # Min-entropy per bit to use instead of H_I (None: use H_I)
TOEPLITZ_BLOCK_SIZE = 1_000_000
TREVISAN_BLOCK_SIZE = 10_000
N_SEQUENCES = 55                # Used in 7.; also decides how many bits to extract
SEQUENCE_LENGTH = 1_000_000

h = H_I if H_MIN_OVERRIDE is None else float(H_MIN_OVERRIDE)
needed_bits = N_SEQUENCES * SEQUENCE_LENGTH


def extract_toeplitz(bits):
    m_b, _ = toeplitz.calculate_output_length(TOEPLITZ_BLOCK_SIZE, h, EPSILON)
    n_blocks = min(bits.size // TOEPLITZ_BLOCK_SIZE, math.ceil(needed_bits / m_b))
    used = bits[:n_blocks * TOEPLITZ_BLOCK_SIZE]
    out, seed, info = toeplitz.toeplitz_extraction_blocks(used, TOEPLITZ_BLOCK_SIZE, h, EPSILON, rng=SEED_RNG)
    return out, seed, dict(info, extractor="toeplitz")


def extract_trevisan(bits):
    extract = trevisan.trevisan_extraction_numba if trevisan.NUMBA_AVAILABLE else trevisan.trevisan_extraction
    outputs, seed, info = [], None, None
    for start in range(0, bits.size - TREVISAN_BLOCK_SIZE + 1, TREVISAN_BLOCK_SIZE):
        block = bits[start:start + TREVISAN_BLOCK_SIZE]
        out, seed, info = extract(block, h, EPSILON, seed=seed, rng=SEED_RNG)   # the first block creates the seed
        outputs.append(out)
        if sum(o.size for o in outputs) >= needed_bits:
            break
    n_blocks = len(outputs)
    info = dict(info, extractor="trevisan", block_size=TREVISAN_BLOCK_SIZE, n_blocks=n_blocks,
                total_output_length=int(sum(o.size for o in outputs)),
                log2_total_epsilon=math.log2(n_blocks * EPSILON))
    return np.concatenate(outputs), seed, info


started = time.time()
if EXTRACTOR == "toeplitz":
    extracted, extraction_seed, extraction_info = extract_toeplitz(raw_bits)
elif EXTRACTOR == "trevisan":
    extracted, extraction_seed, extraction_info = extract_trevisan(raw_bits)
else:
    raise ValueError(f"EXTRACTOR must be 'toeplitz' or 'trevisan' (got: {EXTRACTOR!r})")
extraction_info.update(min_entropy_rate=h, min_entropy_source="H_MIN_OVERRIDE" if H_MIN_OVERRIDE is not None else h_source,
                       extraction_s=time.time() - started)

print(f"extractor  : {EXTRACTOR}, h = {h:.6f}")
print(f"blocks     : {extraction_info['n_blocks']} (log2 of the total error = {extraction_info['log2_total_epsilon']:.1f})")
print(f"output     : {extracted.size:,} bits from {extraction_info['n_blocks'] * extraction_info['block_size']:,} raw bits,"
      f" P(1) = {extracted.mean():.5f} ({extraction_info['extraction_s']:.0f} s)")
if extracted.size < needed_bits:
    print(f"Note: fewer bits than N_SEQUENCES × SEQUENCE_LENGTH = {needed_bits:,}. Increase SHOTS for the full tests.")

if RUN_DIR is not None:
    toeplitz.save_bits(RUN_DIR / "extracted_bits.txt", extracted)
    toeplitz.save_bits(RUN_DIR / "extraction_seed.txt", extraction_seed)
    (RUN_DIR / "extraction_info.json").write_text(json.dumps(extraction_info, indent=2, default=str) + "\n", encoding="utf-8")
    print("Saved:", RUN_DIR / "extracted_bits.txt")

<!-- ## 7. SP 800-22 で検定する -->

## 7. Testing with SP 800-22

<!-- 抽出したビット列を `SEQUENCE_LENGTH` ビットずつの系列に分け（最大 `N_SEQUENCES` 本）、STS で 15 種類の検定を行います（SP800_22 のノートブックの 3.〜4.）。表の読み方と判定の基準は、SP800_22 のノートブックの手法 C と但し書きを参照してください。STS の入力ファイルは一時フォルダに書きます。 -->

We split the extracted bits into sequences of `SEQUENCE_LENGTH` bits (at most `N_SEQUENCES` sequences) and run the 15 tests with STS (3.–4. of the SP800_22 notebook). For how to read the tables and the criteria, see Method C and the caveats in the SP800_22 notebook. The STS input file is written to a temporary folder.

In [ ]:
n_sequences = min(N_SEQUENCES, extracted.size // SEQUENCE_LENGTH)
report = None
if n_sequences == 0:
    print(f"There are fewer than SEQUENCE_LENGTH = {SEQUENCE_LENGTH:,} extracted bits, so the tests are skipped.")
elif not sp22.STS_AVAILABLE:
    print("STS is not available, so the tests are skipped.")
else:
    sequences = sp22.split_sequences(extracted, SEQUENCE_LENGTH, n_sequences)
    sts_input = sp22.write_sts_input(Path(tempfile.mkdtemp(prefix="sts_")) / "extracted.txt", sequences)
    result_dir = sp22.run_sts(sts_input["path"], SEQUENCE_LENGTH, n_sequences,
                              out_dir=None if RUN_DIR is None else RUN_DIR / "sts_results")
    report = sp22.parse_sts_report(result_dir)
    print("Minimum pass count for criterion 1 (passes, s):", report["min_pass"],
          "/ Random Excursions:", report["min_pass_excursion"])
    sp22.show_sts_summary({"extracted": report})
    flagged_rows = sp22.show_flagged_rows(report)
    if RUN_DIR is not None:
        print("Saved:", sp22.save_sts_csv(RUN_DIR / "sts_report.csv", {"extracted": report}))

<!-- ## 8. 結果をまとめる -->

## 8. Summary

<!-- 各段階の主な値を 1 つの表にまとめます。`SAVE_RESULTS = True` なら、同じ内容を `summary.json` に保存します。 -->

The main values of each step are collected in one table. With `SAVE_RESULTS = True`, the same content is saved to `summary.json`.

In [ ]:
summary = {
    "raw data": f"{raw_info['source']}, {raw_bits.size:,} bits, P(1) = {raw_bits.mean():.5f}"
                + (f" ({CIRCUIT}, {GATE}, {SHOTS:,} shots)" if SOURCE == "simulation" else f" ({RAW_FILE})"),
    "min-entropy per bit": f"H_I = {H_I:.6f} ({h_source}, first {samples.size:,} bits)",
    "extraction": f"{EXTRACTOR}, h = {h:.6f}, {extraction_info['n_blocks']} blocks, "
                  f"log2(total error) = {extraction_info['log2_total_epsilon']:.1f}",
    "extracted bits": f"{extracted.size:,} bits, P(1) = {extracted.mean():.5f}",
}
if report is not None:
    flagged = sum(r["proportion_flag"] or r["uniformity_flag"] for r in report["rows"])
    summary["SP 800-22"] = f"{n_sequences} sequences × {SEQUENCE_LENGTH:,} bits, rows with *: {flagged} / {len(report['rows'])}"
else:
    summary["SP 800-22"] = "not run"

display(Markdown("| step | result |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in summary.items())))
if RUN_DIR is not None:
    (RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    print("Saved:", RUN_DIR / "summary.json")